# `Datascience` Tricks and Techniques

## Removing Duplicates
Sometimes a data set has duplicate rows for whatever reason. You don't want to leave these rows in your analysis because they will bias the results, and if you use any machine learning methods, such as KNN, you might get the same data in both your training and test datasets, inflating model scores.

In [34]:
from datascience import *
import numpy as np

### Demo table
A club sign-up sheet. Ana signed up twice (an exact duplicate), and Ben signed up twice with **different** majors (a conflict).

In [35]:
signups = Table().with_columns(
    'name',    make_array('Ana', 'Ben', 'Ana', 'Cy', 'Ben', 'Dee', 'Eli'),
    'section', make_array(1, 2, 1, 1, 2, 3, 3),
    'major',   make_array('Biology', 'Physics', 'Biology', 'Chemistry', 'Math', 'Biology', 'Physics'),
    'hours',   make_array(4, 6, 4, 2, 6, 5, 8),
    'date registered', make_array('2026-10-01','2026-10-03','2026-10-21','2026-11-01','2026-11-05','2026-11-07','2026-11-07')
)
signups

name,section,major,hours,date registered
Ana,1,Biology,4,2026-10-01
Ben,2,Physics,6,2026-10-03
Ana,1,Biology,4,2026-10-21
Cy,1,Chemistry,2,2026-11-01
Ben,2,Math,6,2026-11-05
Dee,3,Biology,5,2026-11-07
Eli,3,Physics,8,2026-11-07


### Remove exact duplicate rows with `group`
Grouping by **every** column gives one row per distinct row, plus a `count` column. Drop `count` and the duplicates are gone.

Note: `group` sorts the rows, so the result is not in the original order.

In [36]:
signups.group(signups.labels)

name,section,major,hours,date registered,count
Ana,1,Biology,4,2026-10-01,1
Ana,1,Biology,4,2026-10-21,1
Ben,2,Math,6,2026-11-05,1
Ben,2,Physics,6,2026-10-03,1
Cy,1,Chemistry,2,2026-11-01,1
Dee,3,Biology,5,2026-11-07,1
Eli,3,Physics,8,2026-11-07,1


In [37]:
unique = signups.group(signups.labels).drop('count')
unique

name,section,major,hours,date registered
Ana,1,Biology,4,2026-10-01
Ana,1,Biology,4,2026-10-21
Ben,2,Math,6,2026-11-05
Ben,2,Physics,6,2026-10-03
Cy,1,Chemistry,2,2026-11-01
Dee,3,Biology,5,2026-11-07
Eli,3,Physics,8,2026-11-07


In [38]:
signups.num_rows, unique.num_rows

(7, 7)

### Finding conflicts: group on some of the columns
Group on the columns that *should* identify a row (here `name` and `section`). A count above 1 means the same person appears with different values in the other columns.

In [39]:
unique.group(make_array('name', 'section')).where('count', are.above(1))

name,section,count
Ana,1,2
Ben,2,2


## Custom data filters: using `where` with your own function
If the second argument to `where` is a function, `where` keeps the rows for which that function returns `True`.
This is not new machinery: the `are.` predicates are just functions.

In [40]:
test = are.above(5)    # are.above(5) returns a function
test(8), test(3)

(True, False)

So any function that takes one value and returns `True` or `False` can go in the same place, which lets you filter on conditions `are.` doesn't cover.

For example, suppose you want just the rows where the student's name starts with a vowel. (Don't ask why, just go with me here.)

In [41]:
# Create the function
def starts_with_vowel(name):
    return name[0] in 'AEIOU'

In [42]:
# Test the function
starts_with_vowel('Harry')

False

In [43]:
starts_with_vowel('Isabelle')

True

Use the function in a where filter in place of an `are` predicate.

In [44]:
unique.where('name', starts_with_vowel)

name,section,major,hours,date registered
Ana,1,Biology,4,2026-10-01
Ana,1,Biology,4,2026-10-21
Eli,3,Physics,8,2026-11-07


In [45]:
# Here is an example of filtering on just rows with an even number of hours.
def is_even(n):
    return n % 2 == 0

unique.where('hours', is_even)

name,section,major,hours,date registered
Ana,1,Biology,4,2026-10-01
Ana,1,Biology,4,2026-10-21
Ben,2,Math,6,2026-11-05
Ben,2,Physics,6,2026-10-03
Cy,1,Chemistry,2,2026-11-01
Eli,3,Physics,8,2026-11-07


Filtering on a list of allowed values:

In [46]:
sciences = make_array('Biology', 'Chemistry', 'Physics')

def is_science(major):
    return major in sciences

unique.where('major', is_science)

name,section,major,hours,date registered
Ana,1,Biology,4,2026-10-01
Ana,1,Biology,4,2026-10-21
Ben,2,Physics,6,2026-10-03
Cy,1,Chemistry,2,2026-11-01
Dee,3,Biology,5,2026-11-07
Eli,3,Physics,8,2026-11-07


## Extracting part of a date
Suppose we want to add a column that just has the month the student registered. We need to split the date.

In [47]:
# Splitting a string
d = '2025-08-13'
d.split('-')

['2025', '08', '13']

In [48]:
# The month is the second element and Python starts counting at zero.
month = d.split('-')[1]
month

'08'

In [55]:
# If we want the month as a number instead of a string
month = int(d.split('-')[1])
month

8

In [56]:
# Create a function and test it
def get_month(date):
    return int(date.split('-')[1])

get_month(d)

8

In [57]:
# Original data table
signups

name,section,major,hours,date registered
Ana,1,Biology,4,2026-10-01
Ben,2,Physics,6,2026-10-03
Ana,1,Biology,4,2026-10-21
Cy,1,Chemistry,2,2026-11-01
Ben,2,Math,6,2026-11-05
Dee,3,Biology,5,2026-11-07
Eli,3,Physics,8,2026-11-07


In [58]:
# Applying the function returns and array
signups.apply(get_month, 'date registered')

array([10, 10, 10, 11, 11, 11, 11])

In [59]:
# add this array as a new column to our table
signups.with_column('month', signups.apply(get_month, 'date registered'))

name,section,major,hours,date registered,month
Ana,1,Biology,4,2026-10-01,10
Ben,2,Physics,6,2026-10-03,10
Ana,1,Biology,4,2026-10-21,10
Cy,1,Chemistry,2,2026-11-01,11
Ben,2,Math,6,2026-11-05,11
Dee,3,Biology,5,2026-11-07,11
Eli,3,Physics,8,2026-11-07,11
